# 🗄️ Chapter 15: Data Engineering

📖 Book: Chapter 15, pp. 620–660 · ⏱️ 2–3 hours · 🧱 Needs: Ch 1, 13 · Level: Advanced

---

> **🎯 Key idea:** Pass the Laptop Test: chain `yield` generators for `O(1)` memory ETL, isolate bad rows to dead-letter queues, and use `boto3` + Parquet/Arrow.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `boto`/`boto3` + CSV  
> **🆕 Now (2026):** `boto3` only + **Parquet/Arrow** (`pyarrow`)  
> **Why:** Legacy `boto` is dead; Parquet is columnar

> **📜 Then (book, ~2020):** Cron / Jenkins scripts  
> **🆕 Now (2026):** **Airflow / Dagster / Prefect**  
> **Why:** Retries, lineage & backfills built in

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A 4 GB CSV export arrives daily, and calling `pandas.read_csv()` on a 16 GB laptop or Lambda freezes the machine with an OOM error. Even worse, one malformed row in row 900,000 aborts the whole batch. You learn lazy `yield` pipelines, dead-letter isolation, `io.StringIO` S3 buffers, and modern Parquet/Arrow.

### Concept: The Laptop Test & `yield` ETL
**What:** Pandas 5–10x RAM rule + lazy pipelines  
**Why:** Loading multi-GB CSVs crashes with OOM  
**How:** Stream line-by-line via `yield`; sample via `shuf`  
**When:** Any dataset > ~1–2 GB

📖 *Book listing:* Lazy `yield` Pipeline & `shuf` Sampling (pp. 624–635)

**Plain-language analogy:** Loading a 4 GB CSV into Pandas is like pouring a swimming pool into a bathtub (5–10x RAM). A `yield` generator pipeline is a garden hose—water flows through one cup at a time.

**Walkthrough:** The book applies the Laptop Test (Pandas 5–10x RAM), samples large TSVs with `shuf -n 100000`, and chains `yield` generators to transform lines in `O(1)` memory.

### Concept: Dead-Letter & `io.StringIO`
**What:** Isolate bad rows & write S3 in memory  
**Why:** 1 corrupt row shouldn't abort a 1M-row batch  
**How:** Catch parse errors into `dead_letter`; stream CSV via `StringIO`  
**When:** Serverless Lambda/S3 ETL

📖 *Book listing:* Serverless DynamoDB -> SQS -> Comprehend -> S3 (pp. 645–659)

**Plain-language analogy:** A dead-letter queue is like a post office's 'return to sender' bin: one unreadable envelope doesn't stop the rest of the mail truck.

**Walkthrough:** The book triggers a producer Lambda on a CloudWatch cron to push DynamoDB items to SQS, and a consumer Lambda that runs Comprehend sentiment analysis and writes CSV via `io.StringIO` to S3.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Chained lazy generator pipeline
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Chained lazy generator pipeline
import io

def read_lines_lazily(stream):
    for line in stream:
        if cleaned := line.strip():
            yield cleaned

raw = io.StringIO('POD-A\n\nPOD-B\nPOD-C\n')
print('Streamed:', list(line.lower() for line in read_lines_lazily(raw)))

Streamed: ['pod-a', 'pod-b', 'pod-c']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Laptop Test RAM estimator
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Laptop Test RAM estimator
def fits_in_laptop_ram(file_size_gb: float, ram_gb: float = 16.0, multiplier: float = 8.0) -> bool:
    return (file_size_gb * multiplier) < (ram_gb * 0.7)

print('1 GB CSV fits 16 GB RAM:', fits_in_laptop_ram(1.0), '| 3 GB CSV:', fits_in_laptop_ram(3.0))

1 GB CSV fits 16 GB RAM: True | 3 GB CSV: False


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Deduplicate records by ID preserving order
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Deduplicate records by ID preserving order
def dedup_by_id(records: list[dict]) -> list[dict]:
    seen: set = set()
    out: list[dict] = []
    for r in records:
        if r['id'] not in seen:
            seen.add(r['id'])
            out.append(r)
    return out

print('Deduped:', dedup_by_id([{'id': 1, 'v': 'a'}, {'id': 1, 'v': 'b'}, {'id': 2, 'v': 'c'}]))

Deduped: [{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'c'}]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Resilient ETL + Dead-Letter & `asyncio` Streaming

#### Example 4: Intermediate Example 1: Resilient CSV ETL with dead-letter isolation
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Resilient CSV ETL with dead-letter isolation
import csv, io

def run_etl(csv_stream) -> dict:
    clean, dead_letter = [], []
    for row in csv.DictReader(csv_stream):
        try:
            val = int(row['bytes'])
            if val < 0:
                raise ValueError('negative bytes')
            clean.append({'service': row['service'].strip().lower(), 'bytes': val})
        except (KeyError, ValueError, TypeError) as err:
            dead_letter.append({'row': row, 'error': str(err)})
    return {'clean': clean, 'dead_letter': dead_letter}

print('ETL:', run_etl(io.StringIO('service,bytes\nAPI,1024\nAuth,BAD\nDB,2048\n')))

ETL: {'clean': [{'service': 'api', 'bytes': 1024}, {'service': 'db', 'bytes': 2048}], 'dead_letter': [{'row': {'service': 'Auth', 'bytes': 'BAD'}, 'error': "invalid literal for int() with base 10: 'BAD'"}]}


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Async streaming + boto3 S3 StringIO
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Async streaming + boto3 S3 StringIO
# Requires: pip install boto3 pandas pyarrow
# WARNING: Real S3/DynamoDB/Comprehend calls incur AWS charges. Clean up buckets/tables afterward!
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def emit_event(i: int) -> dict:
    await asyncio.sleep(0.005)
    return {'id': i, 'status': 'ok'}

async def stream_batch(n: int) -> list[dict]:
    return await asyncio.gather(*(emit_event(i) for i in range(n)))

with ThreadPoolExecutor(1) as ex:
    batch = ex.submit(asyncio.run, stream_batch(3)).result()
print('Async stream batch:', batch)

Async stream batch: [{'id': 0, 'status': 'ok'}, {'id': 1, 'status': 'ok'}, {'id': 2, 'status': 'ok'}]


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **In-Memory CSV Aggregator** ready for S3 `.put(Body=...)`.

#### Example 6: import csv, io
*Expected output:* Printed labelled output below.

In [6]:
import csv, io

def summarize_to_csv_buffer(records: list[dict]) -> str:
    totals: dict[str, int] = {}
    for r in records:
        totals[r['service']] = totals.get(r['service'], 0) + r['bytes']
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(['service', 'total_bytes'])
    for svc, total in sorted(totals.items()):
        w.writerow([svc, total])
    return buf.getvalue()

print(summarize_to_csv_buffer([{'service': 'api', 'bytes': 100}, {'service': 'api', 'bytes': 250}]))

service,total_bytes
api,350



**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: Unix shuf random sampling on a temp TSV
import shutil, subprocess, tempfile
from pathlib import Path
if shutil.which('shuf'):
    with tempfile.TemporaryDirectory() as tmp:
        tsv = Path(tmp) / 'data.tsv'
        tsv.write_text('a\t1\nb\t2\nc\t3\nd\t4\n', encoding='utf-8')
        sample = subprocess.run(['shuf', '-n', '2', str(tsv)], capture_output=True, text=True, check=True).stdout.splitlines()
        print('shuf -n 2 sample:', sample)
else:
    print('shuf not installed; skipped')

shuf -n 2 sample: ['d\t4', 'b\t2']


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Calling `.readlines()` on GB files** -> Exhausts RAM immediately. ✅ Iterate `for line in f:`.
- ❌ **`asyncio.run()` inside Jupyter** -> `RuntimeError: loop already running`. ✅ Use `await` or worker thread.
- ❌ **Crashing batch on 1 bad row** -> Zero rows processed. ✅ Route bad rows to dead-letter.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement batch_items and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement batch_items and run to verify!)
def _run_starter_checks():
    assert list(batch_items(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]]
def batch_items(iterable, size: int):
    raise NotImplementedError('Implement batch_items')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement batch_items


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** What is the 'Laptop Test'?

<details>
<summary>✅ Answer 1</summary>

Pandas uses ~5–10x file size in RAM; datasets exceeding laptop RAM require Big Data tooling.

</details>

**Q2:** Why use `io.StringIO` in AWS Lambda?

<details>
<summary>✅ Answer 2</summary>

Builds CSVs in memory for S3 upload without touching Lambda's limited `/tmp` disk.

</details>

**Q3:** What is a dead-letter record?

<details>
<summary>✅ Answer 3</summary>

Isolating malformed rows so one bad record doesn't crash the whole pipeline.

</details>

**Q4:** Why prefer Parquet/Arrow over CSV for data lakes?

<details>
<summary>✅ Answer 4</summary>

Columnar compression and typed schemas make queries faster and storage smaller.

</details>

**Q5:** Why use `shuf -n 100000`?

<details>
<summary>✅ Answer 5</summary>

Samples a small random subset of a massive file for fast local prototyping.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: Accumulate `chunk = []`; yield when `len(chunk) == size`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import io

def batch_items(iterable, size: int):
    chunk = []
    for item in iterable:
        chunk.append(item)
        if len(chunk) == size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk

def process_pipeline(csv_stream) -> dict:
    res = run_etl(csv_stream)
    return {'csv_output': summarize_to_csv_buffer(res['clean']), 'processed': len(res['clean']), 'failed': len(res['dead_letter'])}

assert list(batch_items(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]]
out = process_pipeline(io.StringIO('service,bytes\napi,50\napi,150\ndb,-5\n'))
assert out['processed'] == 2 and out['failed'] == 1
print('All Chapter 15 assertions PASSED!')

All Chapter 15 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `io.StringIO()` — In-memory CSV buffer for S3 (e.g. `s3.put(Body=buf.getvalue())`)
- `shuf -n 100000 big.tsv` — Local random sampling (e.g. Prototype fast before Spark)
- `yield` generator chain — `O(1)` memory ETL (e.g. `(x.lower() for x in reader)`)

### ✅ What You Can Now Do
- Apply the Laptop Test and sample huge TSVs with Unix `shuf` before scaling out.
- Chain `yield` generators for `O(1)` memory ETL and isolate bad rows into dead-letter lists.
- Run `asyncio` ingestion safely and format in-memory CSVs with `io.StringIO` for `boto3` S3 uploads.

### ➡️ Next Chapter
Chapter 16 wraps up the course with real-world DevOps war stories, structured postmortems, and profiling tools.